# Tutorial de 5 minutos

## OPeNDAP: la visión
La visión original de [OPeNDAP](https://www.opendap.org/) ([Cornillion, et al 1993](https://zenodo.org/records/10610992)) es el democratizar el acceso remoto a datos, estableciendo las equivalencias

$ \;\;\;\;\;\;\;\;\;\;\;\;\;\;\;\;\;\;\;\;\;\;\;\;\;\;\;\;\;\;\;\;\;\;\;\;\; \boxed{\text{URL} \approx \text{Conjunto de datos remoto} }$

$ \;\;\;\;\;\;\;\;\;\;\;\;\;\;\;\;\;\; \boxed{\text{URL + Restricciones} \approx \text{Subconjunto del conjunto de datos remoto}} $

Eso llevó al desarrollo del protocolo `DAP2` (antes conocido como `DODS`). Actualmente, los servidores <span style='color:#ff6666'>**OPeNDAP**</span> y Unidata implementan el protocolo moderno y más amplio <span style='color:#0066cc'>**DAP4**</span> (consulta la [especificación DAP4](https://opendap.github.io/dap4-specification/DAP4.html#_how_dap4_differs_from_dap2)), para seguir habilitando la visión original de OPeNDAP.

## Pydap

La lógica interna de `PyDAP` permite construir expresiones de restricción para cada URL de forma interactiva, de tal manera que el usuario ya no debe de hacerlo manualmente. Esta es una gran ventaja en el caso cuanto el usuario tiene a su mano 100s de URLs de los cuales solo una fraccion de las datos son de relevancia. Aun así, una comprensión básica del uso de expresiones de restricción resulta útil al agregar múltiples archivos y puede conducir a flujos de trabajo más eficientes.


### Objetivos:


- Demostrar cómo especificar el protocolo <span style='color:#0066cc'>**DAP4**</span> al servidor remoto.
- Demostrar el uso de `Xarray` con `PyDAP` para la exploracion de datos remotos de la NASA.
- Demostrar distintas formas de usar expresiones de restricción (`CE`s), y cómo <span style='color:#0066cc'>**asegurar que el servidor remoto es el que implementa el segmento de subconjunto (slice en ingles)**</span>, de forma `próxima a los datos`, sin pérdida de rendimiento del lado del cliente.

### Requisitos

- Archivos de datos cientificos detrás de un servidor de OPeNDAP que implemente el protocolo <span style='color:#0066cc'>**DAP4**</span>. Por ejemplo, el servidor de pruebas: http://test.opendap.org/opendap/.
- pydap
- xarray
- python>=3.12

```{note}
La gran mayoría de los servidores OPeNDAP de NASA implementan el protocolo DAP4.
```


In [ ]:
from pydap.client import open_url, consolidate_metadata, create_session
import xarray as xr
import numpy as np

In [ ]:
# create a session to inspect downloads. cache_name must have `debug`
session = create_session(use_cache=True, cache_kwargs={"cache_name":'data/debug_case1'})
session.cache.clear()

## Segmento de subconjunto de dos archivos distintos.

Considera el caso en el que el servidor remoto permite el acceso a dos archivos distintos que forman parte de una misma simulacion, or proyecto. En este case usaremos [coads_climatology](http://test.opendap.org/opendap/data/nc/coads_climatology.nc.dmr.html) y [coads_climatology2](http://test.opendap.org/opendap/data/nc/coads_climatology.nc.dmr.html). Estos dos archivos de datos comparten dimensiones espaciales idénticas, se pueden agregar en el tiempo, y comparten casi todas las mismas variables.

```{note}
Es importante comprobar siempre que los conjuntos de datos se puedan agregar. `PyDAP` y `Xarray` tienen lógica interna para verificar si dos o más conjuntos de datos se pueden concatenar. 
```

<span style='color:#0066cc'>**Un paso importante será usar Expresiones de Restricción (CEs) para asegurar que solo se concatenen las variables de interés**<span style='color:black'>.

```{warning}
Uno de estos archivos tiene variables extra que no están presentes en el otro archivo y que serán descartadas mediante el uso de CEs.
```


In [ ]:
urls = ["http://test.opendap.org/opendap/data/nc/coads_climatology.nc", "http://test.opendap.org/opendap/data/nc/coads_climatology2.nc"]
dap4_urls = [url.replace("http","dap4") for url in urls]

# constraint expression
dap4_CE = "?dap4.ce=" + ";".join(["/SST", "/COADSX", "/COADSY", "/TIME"])

# Final list of OPeNDAP URLs
dap4ce_urls =[url+dap4_CE for url in dap4_urls]
print("====================================================\nThe following are the DAP4 OPeNDAP URLs \n", dap4ce_urls)

```{note}
**P: ¿Por qué usar `CE`s cuando `Xarray` tiene un método `.drop_variables`?** Porque `Xarray` primero necesita analizar TODOS los metadatos asociados con el archivo remoto para después descartar las variables. En algunos archivos remotos puede haber miles de variables. Entonces `Xarray` las analizaría todas y luego las descartaría. Con la `CE`, el servidor envía metadatos restringidos asociados solo con las variables deseadas.
```


```{warning}
`Xarray` espera la presencia de dimensiones en los metadatos. Al construir `CE`s, el usuario debe asegurarse de incluir en la CE todas las dimensiones asociadas con las variables de interés. En el ejemplo anterior, `COASX`, `COADSY` y `TIME` son las dimensiones de `SST`.
```


### <span style='color:#0066cc'>**Consolidate Metadata acelera la generación del Dataset**<span style='color:black'>.


In [ ]:
consolidate_metadata(dap4ce_urls, session=session, concat_dim="TIME")

```{note}
`consolidate_metadata(dap4_urls, concat_dim='...', session=session)` descarga las dimensiones del archivo remoto y las almacena como SQLite para reutilizarlas. El objeto de sesión se convierte en una forma de autenticar y actuar como gestor de base de datos. Esta práctica puede producir una mejora de rendimiento de flujos de trabajo aproximadamente 10 a 100 veces más rápidos.
```


### Usar la lógica de Xarray para descargar datos.


In [ ]:
ds = xr.open_mfdataset(
    dap4ce_urls, 
    engine='pydap',
    concat_dim='TIME',
    session=session,
    combine="nested",
    parallel=True,
    decode_times=False,
    chunks={},
)
ds

In [ ]:
ds['SST']

```{note}
El chunking de `SST` implica que el arreglo completo dentro de cada archivo es un solo chunk. Esta es una interpretación estándar que `Xarray` hace de las URLs de `OPeNDAP`. ¿Qué ocurre si descargamos un solo punto espacial desde un único archivo remoto?
```


In [ ]:
session.cache.clear()

In [ ]:
%%time
ds['SST'].isel(TIME=0, COADSX=0, COADSY=0).load() # this should download a single point one of the files

In [ ]:
print("====================================== \n Request sent to the Remote Server:\n ", session.cache.urls()[0].split("?")[-1].split("&dap4.checksum")[0].replace("%5B","[").replace("%5D","]").replace("%3A",":").replace("%2F","/"), "\n====================================== ")

### <span style='color:#0066cc'>**Toda la variable completa se descarga innecesariamente<span style='color:black'>** !!

Lo ideal seria que Xarray mande la siguente `solicitud` (request en ingles) (en la expresión de restricción) al servidor remoto:

```python
dap4.ce=/SST[0:1:0][0:1:0][0:1:0]
```
El metodo `xr.open_mfdataset` no pasa el argumento de segmentar el subconjunto al servidor para cada archivo remoto. En su lugar, el metodo demostrate arriba descarga todo el chunk (es decir, el arreglo de datos) en una sola `solicitud`, lo subdivide ya estando en la computadora, y luego agrega los datos. Este proceso no es muu eficiente y conlleva a descargar innecesariamente variables completas.


### <span style='color:#0066cc'>**Cómo pasar el la segmentacion del archivo desde Xarray al servidor remoto?<span style='color:black'>**


**La respuesta es definir `chunks` al momento de crear el Xarray.Dataset**. `chunks` debe definir, por variable un tamano que **debera coincidir con el tamaño esperado de tu subconjunto**. Así el subconjunto se procesará en una sola solicitud por archivo remoto.

```{warning}
Si fragmentas el dataset con un tamaño menor que tu descarga esperada, activarás muchas descargas por archivo remoto, obligando a `Xarray` a hacer trabajo extra para ensamblar los datos.
```


In [ ]:
# consolidate metadata again, since the cached metadata was cleared before
session = create_session(use_cache=True, cache_kwargs={"cache_name":'data/debug_case2'})
session.cache.clear()
consolidate_metadata(dap4ce_urls, session=session, concat_dim="TIME")


In [ ]:
# For a single element in all dimensions, the expected size of the download is:
expected_sizes = {"TIME":1, "COADSX":1, "COADSY":1}

In [ ]:
%%time
ds = xr.open_mfdataset(
    dap4ce_urls, 
    engine='pydap',
    concat_dim='TIME',
    session=session,
    combine="nested",
    parallel=True,
    decode_times=False,
    chunks=expected_sizes,
)
session.cache.clear()

In [ ]:
ds['SST'] # inspect chunks before download

In [ ]:
%%time
ds['SST'].isel(TIME=0, COADSX=0, COADSY=0).load() # triggers download of an individual chunk

In [ ]:
print("====================================== \n Request sent to the Remote Server:\n ", session.cache.urls()[0].split("?")[-1].split("&dap4.checksum")[0].replace("%5B","[").replace("%5D","]").replace("%3A",":").replace("%2F","/"), "\n====================================== ")

### Advertencia cuando se trabaja con `chunks`

Ahora descargamos exactamente lo que solicitamos. Sin embargo, en algunos escenarios el tiempo de descarga puede ser 10 veces más lento en comparación con el caso en que solicitamos más datos. La razón puede atribuirse al número de chunks que genera Dask al orquestrar el proceso.


* Sin `chunks`. Descarga todo el arreglo del archivo. 2 chunks en 5 graficas dask (uno por archivo).`
* Con `chunk`. Descargar solo el elemento deseado de un archivo. 388800 chunks en 5 graficas de dask`.

Idealmente, Dask, el gestor de `chunks`, solo debería activar la descarga de un único chunk. Sin embargo, se crearon `388800` para asegurar que el rebanado se pasara al servidor. Esto a veces puede causar que el proceso tarde mas para el usuario.

En el escenario anterior fuimos al extremo de definir `chunks` con el elemento mas pequeno posible. A continuacion demostraremos algo mas eficiente para este caso, <span style='color:#0066cc'>**subdividiendo en la dimension a lo largo the la dimension `COARDSY` (en ambos archivos)**</span>.


In [ ]:
session = create_session(use_cache=True, cache_kwargs={"cache_name":'data/debug_case3'})
session.cache.clear()

consolidate_metadata(dap4ce_urls, session=session, concat_dim="TIME")

In [ ]:
download_sizes = {"COADSY":1} # note that we will subset across all time

In [ ]:
%%time
ds = xr.open_mfdataset(
    dap4ce_urls, 
    engine='pydap',
    concat_dim='TIME',
    session=session,
    combine="nested",
    parallel=True,
    decode_times=False,
    chunks=download_sizes,
)
session.cache.clear()

In [ ]:
ds['SST']

In [ ]:
%%time
ds['SST'].isel(COADSX=0, COADSY=0).load()

In [ ]:
print("====================================== \n Parallel Requests sent to the Remote Server:\n ", [url.split("?")[-1].split("&dap4.checksum")[0].replace("%5B","[").replace("%5D","]").replace("%3A",":").replace("%2F","/") for url in session.cache.urls()], "\n====================================== ")

### ¡Éxito! Una descarga mas rapida y mucho más pequeña que la original!
